# MUSE Benchmark – Machine Unlearning on Gemma 3 1B

This notebook implements the **actual MUSE benchmark** (MUSE-Bench) unlearning
pipeline adapted for **Gemma 3 1B** on a Kaggle T4 GPU.

It uses the official MUSE-Bench code for:
- **Unlearning baselines** – Gradient Ascent (GA), Gradient Ascent + KL Retain (GA-KLR), Task Vectors (TV)
- **Evaluation metrics** – VerbMem (verbatim memorisation), PrivLeak (privacy leakage AUC), KnowMem (knowledge memorisation via QA)

> **Dataset**: `muse-bench/MUSE-News` from HuggingFace  
> **Model**: `google/gemma-3-1b-it`  
> **Auth**: HF_TOKEN stored in Kaggle Secrets

---
## 1. Environment Setup
**Important:** We do NOT upgrade `torch` here — Kaggle already ships a compatible torch+torchvision pair. Upgrading torch breaks torchvision.

In [ ]:
!pip install -q transformers datasets bitsandbytes accelerate rouge-score scikit-learn scipy tqdm

## 2. Hugging Face Authentication
Gemma requires accepting the model licence. Your `HF_TOKEN` must be stored in **Kaggle Secrets** (Add-ons → Secrets).

In [ ]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("✅ Authenticated with Hugging Face successfully!")
except Exception as e:
    print("⚠️  Could not find HF_TOKEN in Kaggle Secrets")
    print("Please set it up via Add-ons -> Secrets to download Gemma.", e)

## 3. Imports & Configuration

In [ ]:
import os, json, re, zlib, gc, copy
from typing import List, Dict, Tuple, Literal
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import transformers
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    Trainer, TrainingArguments,
    BitsAndBytesConfig
)
from datasets import load_dataset
from rouge_score import rouge_scorer
from sklearn.metrics import auc as get_auc, roc_curve as get_roc_curve
from scipy.stats import bootstrap
from tqdm.auto import tqdm

# ── Configuration ───────────────────────────────────────────────────────
MODEL_NAME    = "google/gemma-3-1b-it"
CORPUS        = "news"                       # 'news' or 'books'
MAX_LEN       = 512                          # reduced for T4 16 GB VRAM
EPOCHS        = 5
LR            = 1e-5
BATCH_SIZE    = 1                            # per-device; single T4
ALPHA_TV      = 5.0                          # task-vector scaling
DATA_DIR      = "./muse_data"                # local data cache

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  Model: {MODEL_NAME}  |  Corpus: {CORPUS}")

## 4. Utility Helpers
*(Adapted from `muse_bench/utils.py` and `muse_bench/baselines/baselines/utils.py`)*

In [ ]:
# ── I/O helpers ─────────────────────────────────────────────────────────
def read_json(fpath: str):
    with open(fpath, 'r') as f:
        return json.load(f)

def write_json(obj, fpath: str):
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    with open(fpath, 'w') as f:
        json.dump(obj, f)

def write_text(obj: str, fpath: str):
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    with open(fpath, 'w') as f:
        f.write(obj)

def write_csv(obj, fpath: str):
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    pd.DataFrame(obj).to_csv(fpath, index=False)

# ── Model helpers ───────────────────────────────────────────────────────
def load_model(model_dir: str, **kwargs):
    return AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0",
        **kwargs
    )

def load_tokenizer(tokenizer_dir: str, add_pad_token=True, use_fast=True):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=use_fast)
    if add_pad_token:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def load_model_and_tokenizer(model_dir, tokenizer_dir=None, add_pad_token=True):
    model = load_model(model_dir)
    tok_dir = tokenizer_dir if tokenizer_dir else model_dir
    tokenizer = load_tokenizer(tok_dir, add_pad_token)
    return model, tokenizer

# ── Tensor helpers ──────────────────────────────────────────────────────
def pad_or_trim_tensor(tensor, target_length, padding_value=0):
    cur = tensor.size(0)
    if cur < target_length:
        pad = torch.full((target_length - cur,), padding_value, dtype=tensor.dtype)
        return torch.cat((tensor, pad))
    return tensor[:target_length]

print("✅ Utilities loaded.")

## 5. Download MUSE-News Dataset
*(Adapted from `muse_bench/load_data.py`)* — pulls data from `muse-bench/MUSE-News` on HuggingFace.

In [ ]:
def download_muse_data(corpus='news', data_dir=DATA_DIR):
    """Download all MUSE splits from HuggingFace and cache as local JSON/TXT."""
    Corpus = corpus.capitalize()  # 'News' or 'Books'
    hf_name = f"muse-bench/MUSE-{Corpus}"
    print(f"Downloading {hf_name} ...")

    # ── KnowMem QA splits ──
    for split in ['forget_qa', 'retain_qa', 'forget_qa_icl', 'retain_qa_icl']:
        try:
            data = load_dataset(hf_name, 'knowmem', split=split)
            knowmem = [
                {'question': q, 'answer': a}
                for q, a in zip(data['question'], data['answer'])
            ]
            write_json(knowmem, f"{data_dir}/{corpus}/knowmem/{split}.json")
            print(f"  ✓ knowmem/{split} ({len(knowmem)} samples)")
        except Exception as e:
            print(f"  ✗ knowmem/{split} — {e}")

    # ── VerbMem forget ──
    try:
        data = load_dataset(hf_name, 'verbmem', split='forget')
        verbmem = [
            {'prompt': p, 'gt': g}
            for p, g in zip(data['prompt'], data['gt'])
        ]
        write_json(verbmem, f"{data_dir}/{corpus}/verbmem/forget.json")
        print(f"  ✓ verbmem/forget ({len(verbmem)} samples)")
    except Exception as e:
        print(f"  ✗ verbmem/forget — {e}")

    # ── PrivLeak splits ──
    for split in ['forget', 'retain', 'holdout']:
        try:
            privleak = load_dataset(hf_name, 'privleak', split=split)['text']
            write_json(list(privleak), f"{data_dir}/{corpus}/privleak/{split}.json")
            print(f"  ✓ privleak/{split} ({len(privleak)} samples)")
        except Exception as e:
            print(f"  ✗ privleak/{split} — {e}")

    # ── Raw train text (forget / retain / holdout) ──
    for split in ['forget', 'holdout', 'retain1', 'retain2']:
        try:
            raw = load_dataset(hf_name, 'raw', split=split)['text']
            raw = list(raw)
            write_json(raw, f"{data_dir}/{corpus}/raw/{split}.json")
            write_text("\n\n".join(raw), f"{data_dir}/{corpus}/raw/{split}.txt")
            print(f"  ✓ raw/{split} ({len(raw)} documents)")
        except Exception as e:
            print(f"  ✗ raw/{split} — {e}")

    print("✅ Data download complete!")

download_muse_data(CORPUS)

## 6. Dataset Classes
*(Adapted from `muse_bench/baselines/baselines/dataset.py`)*

In [ ]:
class DefaultDataset(Dataset):
    """Tokenise raw text documents into fixed-length input_ids.
    Source: muse_bench/baselines/baselines/dataset.py
    """

    def __init__(self, file_path, tokenizer=None, max_len=MAX_LEN, add_bos_token=True):
        if Path(file_path).suffix == '.json':
            with open(file_path, 'r') as f:
                data = json.load(f)
            if isinstance(data[0], str):
                self.strings = data
            elif isinstance(data[0], dict) and 'text' in data[0]:
                self.strings = [d['text'] for d in data]
                if 'input_ids' in data[0]:
                    self.input_ids = [torch.tensor(d['input_ids']) for d in data]
                    return
            else:
                raise ValueError("Unrecognised JSON format.")

            assert tokenizer is not None
            self.input_ids = []
            for s in self.strings:
                enc = tokenizer(s, add_special_tokens=add_bos_token, return_tensors='pt').input_ids[0]
                enc = pad_or_trim_tensor(enc, max_len, tokenizer.pad_token_id)
                self.input_ids.append(enc)
            return

        # .txt path
        assert Path(file_path).suffix == '.txt'
        with open(file_path, 'r') as f:
            text = f.read()
        tokens = tokenizer(text, add_special_tokens=False, return_tensors='pt').input_ids[0]
        if add_bos_token:
            self.input_ids = [
                F.pad(tokens[i : i + max_len - 1], (1, 0), value=tokenizer.bos_token_id)
                for i in range(0, len(tokens), max_len - 1)
            ]
        else:
            self.input_ids = [
                tokens[i : i + max_len]
                for i in range(0, len(tokens), max_len)
            ]
        if len(self.input_ids[-1]) < max_len:
            self.input_ids[-1] = torch.cat(
                [self.input_ids[-1], self.input_ids[0]], dim=-1
            )[:max_len]
        self.strings = tokenizer.batch_decode(self.input_ids, skip_special_tokens=True)

    def __getitem__(self, idx): return self.input_ids[idx]
    def __len__(self): return len(self.input_ids)

    def get_collate_fn(self):
        def collate_fn(batch):
            batch = torch.stack(batch)
            return {"input_ids": batch, "labels": batch.clone()}
        return collate_fn


class ForgetRetainDataset(DefaultDataset):
    """Paired forget/retain dataset for iterative unlearning.
    Source: muse_bench/baselines/baselines/dataset.py
    """

    def __init__(self, forget_file, tokenizer, retain_file=None,
                 max_len=MAX_LEN, add_bos_token=True):
        self.forget_dataset = DefaultDataset(forget_file, tokenizer, max_len, add_bos_token)
        self.retain_exists = retain_file is not None
        if self.retain_exists:
            self.retain_dataset = DefaultDataset(retain_file, tokenizer, max_len, add_bos_token)
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        if self.retain_exists:
            return (self.forget_dataset[idx],
                    self.retain_dataset[idx % len(self.retain_dataset)])
        return self.forget_dataset[idx], None

    def __len__(self): return len(self.forget_dataset)

    def get_collate_fn(self):
        def collate_fn(batch):
            batch_f = torch.stack([p[0] for p in batch])
            dict_f = {
                "input_ids": batch_f,
                "labels": batch_f.clone(),
                "attention_mask": torch.ones_like(batch_f)
            }
            if self.retain_exists:
                batch_r = torch.stack([p[1] for p in batch])
                dict_r = {
                    "input_ids": batch_r,
                    "labels": batch_r.clone(),
                    "attention_mask": torch.ones_like(batch_r, dtype=torch.bool)
                }
            else:
                dict_r = None
            return dict_f, dict_r
        return collate_fn

print("✅ Dataset classes ready.")

## 7. Iterative Unlearning Baselines
*(Adapted from `muse_bench/baselines/baselines/iterative.py`)*

Supported loss types:
- **`ga`** — Gradient Ascent
- **`ga_klr`** — Gradient Ascent + KL divergence on retain set
- **`npo`** — Negative Preference Optimisation
- **`npo_gdr`** — NPO + Gradient Difference on retain
- **`npo_klr`** — NPO + KL divergence on retain

In [ ]:
class IterativeUnlearner(Trainer):
    """Custom Trainer implementing various unlearning loss functions.
    Source: muse_bench/baselines/baselines/iterative.py
    Originally from: https://github.com/locuslab/tofu/blob/main/dataloader.py
    """

    def __init__(self, *args, loss_type='ga',
                 ref_model=None, beta=0.1, **kwargs):
        self.loss_type = loss_type
        self.ref_model = ref_model
        self.beta = beta
        if ref_model is not None:
            assert 'po' in self.loss_type or 'kl' in self.loss_type
            ref_model = ref_model.eval()
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, x, return_outputs=False):
        """Multi-objective unlearning loss.
        Source: https://github.com/licong-lin/negative-preference-optimization
        """
        x_f, x_r = x
        outputs_f = model(
            x_f['input_ids'],
            labels=x_f.get('labels', x_f['input_ids'].clone()),
            attention_mask=x_f.get('attention_mask',
                                   torch.ones_like(x_f['input_ids'], dtype=torch.bool))
        )
        loss_f = outputs_f.loss

        if 'gdr' in self.loss_type or 'klr' in self.loss_type:
            outputs_r = model(
                x_r['input_ids'],
                labels=x_r.get('labels', x_r['input_ids'].clone()),
                attention_mask=x_r.get('attention_mask',
                                       torch.ones_like(x_r['input_ids'], dtype=torch.bool))
            )
            loss_r = outputs_r.loss

        if 'klf' in self.loss_type or 'npo' in self.loss_type:
            with torch.no_grad():
                outputs_f_ref = self.ref_model(
                    x_f['input_ids'],
                    labels=x_f.get('labels', x_f['input_ids'].clone()),
                    attention_mask=x_f.get('attention_mask',
                                           torch.ones_like(x_f['input_ids'], dtype=torch.bool))
                )

        if 'klr' in self.loss_type:
            with torch.no_grad():
                outputs_r_ref = self.ref_model(
                    x_r['input_ids'],
                    labels=x_r.get('labels', x_r['input_ids'].clone()),
                    attention_mask=x_r.get('attention_mask',
                                           torch.ones_like(x_r['input_ids'], dtype=torch.bool))
                )

        # ── Compose Loss ──
        loss = 0
        if 'ga' in self.loss_type:
            loss += -loss_f
        elif 'npo' in self.loss_type:
            neg_log_ratio = outputs_f_ref.logits - outputs_f.logits
            loss += -F.logsigmoid(self.beta * neg_log_ratio).mean() * 2 / self.beta
        else:
            raise NotImplementedError(f"Cannot infer loss from: {self.loss_type}")

        if 'gdr' in self.loss_type:
            loss += loss_r
        if 'klr' in self.loss_type:
            kl_r = F.kl_div(outputs_r.logits, outputs_r_ref.logits,
                            reduction='batchmean', log_target=True)
            loss += kl_r

        return (loss, outputs_f) if return_outputs else loss

    def prediction_step(self, model, x, prediction_loss_only, ignore_keys=None):
        input_ids, labels, attention_mask = x
        with torch.no_grad():
            outputs = model(input_ids, labels=labels, attention_mask=attention_mask)
        return (outputs.loss, outputs.logits, labels)


def iterative_unlearn(
    model_dir, data_file, out_dir,
    retain_data_file=None, loss_type='ga',
    per_device_batch_size=BATCH_SIZE, epochs=EPOCHS,
    learning_rate=LR, max_len=MAX_LEN,
    tokenizer_dir=None, resume_from_checkpoint=False
):
    """Run iterative unlearning (GA / NPO / GD variants).
    Source: muse_bench/baselines/baselines/iterative.py – unlearn()
    """
    if 'gd' in loss_type:
        assert retain_data_file is not None, "Retain data required for grad-diff."

    model, tokenizer = load_model_and_tokenizer(
        model_dir, processing_class_dir=tokenizer_dir
    )

    ref_model = (
        load_model(model_dir)
        if 'npo' in loss_type or 'kl' in loss_type
        else None
    )

    dataset = ForgetRetainDataset(
        data_file, processing_class=tokenizer,
        retain_file=retain_data_file, max_len=max_len
    )

    training_args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=per_device_batch_size,
        learning_rate=learning_rate,
        save_strategy='epoch',
        num_train_epochs=epochs,
        optim='adamw_torch',
        lr_scheduler_type='constant',
        bf16=True,
        report_to='none',
        gradient_accumulation_steps=4,
        logging_steps=10,
    )

    trainer = IterativeUnlearner(
        model=model, ref_model=ref_model, processing_class=tokenizer,
        train_dataset=dataset, args=training_args,
        data_collator=dataset.get_collate_fn(),
        loss_type=loss_type
    )
    model.config.use_cache = False
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    trainer.save_model(out_dir)
    print(f"✅ Iterative unlearning ({loss_type}) saved to {out_dir}")
    return out_dir

print("✅ Iterative unlearning baselines ready.")

## 8. Task Vector Unlearning
*(Adapted from `muse_bench/baselines/baselines/task_vector.py`)*

Task Arithmetic: `θ_unlearned = θ_pretrained − α · (θ_finetuned − θ_pretrained)`

In [ ]:
class TaskVector:
    """Compute and apply task vectors for unlearning.
    Source: muse_bench/baselines/baselines/task_vector.py
    """
    def __init__(self, pretrained_state_dict=None, finetuned_state_dict=None, vector=None):
        if vector is not None:
            self.vector = vector
        else:
            assert pretrained_state_dict is not None and finetuned_state_dict is not None
            with torch.no_grad():
                self.vector = {}
                for key in pretrained_state_dict:
                    if pretrained_state_dict[key].dtype in [torch.int64, torch.uint8]:
                        continue
                    self.vector[key] = finetuned_state_dict[key] - pretrained_state_dict[key]

    def __neg__(self):
        with torch.no_grad():
            return TaskVector(vector={k: -v for k, v in self.vector.items()})

    def is_nonzero(self):
        return any((self.vector[k] != 0).any() for k in self.vector)

    def apply_to(self, pretrained_model, scaling_coef=1.0, in_place=False):
        with torch.no_grad():
            new_sd = {}
            sd = pretrained_model.state_dict()
            for key in sd:
                if key not in self.vector:
                    print(f"Warning: Key {key} not in task vector")
                    continue
                new_sd[key] = sd[key] + scaling_coef * self.vector[key]
        if in_place:
            pretrained_model.load_state_dict(new_sd, strict=False)
        return new_sd


def finetune_model(model_dir, data_file, out_dir,
                   epochs=EPOCHS, per_device_batch_size=BATCH_SIZE,
                   learning_rate=LR, max_len=MAX_LEN, tokenizer_dir=None):
    """Standard fine-tuning on a text file.
    Source: muse_bench/baselines/baselines/finetune.py
    """
    model, tokenizer = load_model_and_tokenizer(model_dir, processing_class_dir=tokenizer_dir)
    dataset = DefaultDataset(data_file, processing_class=tokenizer, max_len=max_len)

    training_args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=per_device_batch_size,
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        optim='adamw_torch',
        lr_scheduler_type='cosine',
        bf16=True,
        report_to='none',
        gradient_accumulation_steps=4,
        logging_steps=10,
    )
    trainer = Trainer(
        model=model, processing_class=tokenizer,
        train_dataset=dataset, args=training_args,
        data_collator=dataset.get_collate_fn()
    )
    model.config.use_cache = False
    trainer.train()
    trainer.save_model(out_dir)
    print(f"✅ Fine-tuning saved to {out_dir}")
    return out_dir


def task_vector_unlearn(model_dir, out_dir,
                        pt_model_dir=None, ft_model_dir=None,
                        alpha=ALPHA_TV):
    """Task-vector unlearning.
    theta_unlearned = theta_target - alpha * (theta_ft - theta_pt)
    Source: muse_bench/baselines/baselines/task_vector.py – unlearn()
    """
    if pt_model_dir is None or ft_model_dir is None:
        raise ValueError("Task vector requires both pretrained & finetuned model dirs.")

    tv = TaskVector(
        pretrained_state_dict=load_model(pt_model_dir).state_dict(),
        finetuned_state_dict=load_model(ft_model_dir).state_dict()
    )
    if not tv.is_nonzero():
        raise ValueError("Zero task vector — fine-tuning produced no change!")

    neg_tv = -tv
    model = load_model(model_dir)
    new_sd = neg_tv.apply_to(model, scaling_coef=alpha, in_place=False)
    del model; gc.collect(); torch.cuda.empty_cache()

    new_model = load_model(model_dir, state_dict=new_sd)
    os.makedirs(out_dir, exist_ok=True)
    new_model.save_pretrained(out_dir)
    print(f"✅ Task-vector unlearning (alpha={alpha}) saved to {out_dir}")
    del new_model; gc.collect(); torch.cuda.empty_cache()
    return out_dir

print("✅ Task Vector unlearning ready.")

## 9. MUSE Evaluation Metrics
*(Adapted from `muse_bench/metrics/`)*

| Metric | What it measures |
|---|---|
| **VerbMem** | Can the model still reproduce memorised text verbatim? (ROUGE-L) |
| **PrivLeak** | Can a membership-inference attack distinguish forget vs. holdout data? (AUC) |
| **KnowMem** | Can the model answer factual questions about forgotten / retained content? (ROUGE-L) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# RougeEvalLogger  (from muse_bench/metrics/logger.py)
# ══════════════════════════════════════════════════════════════════════════
class RougeEvalLogger:
    def __init__(self):
        self.scorer = rouge_scorer.RougeScorer(
            rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"],
            use_stemmer=False
        )
        self.history = []

    def log(self, prompt, gt, output, question=None):
        score = self.scorer.score(gt, output)
        d = {
            'prompt': prompt, 'gt': gt, 'response': output,
            'rougeL': score['rougeL'].fmeasure,
            'rougeL_recall': score['rougeL'].recall,
            'rouge1': score['rouge1'].fmeasure,
            'rouge1_recall': score['rouge1'].recall,
        }
        if question is not None:
            d['question'] = question
        self.history.append(d)

    def report(self):
        agg = {}
        for key, val in self.history[0].items():
            if isinstance(val, str):
                continue
            vals = [item[key] for item in self.history]
            agg[f"max_{key}"] = max(vals)
            agg[f"mean_{key}"] = sum(vals) / len(vals)
            ci = bootstrap((vals,), np.mean, confidence_level=0.95, method='percentile')
            agg[f"{key}_ci_lo"], agg[f"{key}_ci_hi"] = ci.confidence_interval
        return agg, self.history


# ══════════════════════════════════════════════════════════════════════════
# VerbMem  (from muse_bench/metrics/verbmem.py)
# ══════════════════════════════════════════════════════════════════════════
def eval_verbmem(model, tokenizer, prompts, gts, max_new_tokens=128):
    """Verbatim memorisation: generate continuations and compare via ROUGE."""
    logger = RougeEvalLogger()
    for prompt, gt in tqdm(list(zip(prompts, gts)), desc="VerbMem"):
        input_ids = tokenizer(prompt, return_tensors='pt',
                              add_special_tokens=True).input_ids
        gt_ids = tokenizer(gt, return_tensors='pt',
                           add_special_tokens=True).input_ids[:, :max_new_tokens]
        output_ids = model.generate(
            input_ids.to(model.device),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
        output_ids = output_ids[:, len(input_ids[0]):]
        output = tokenizer.batch_decode(output_ids, skip_special_tokens=True,
                                        clean_up_tokenization_spaces=True)[0]
        gt_short = tokenizer.batch_decode(gt_ids, skip_special_tokens=True,
                                          clean_up_tokenization_spaces=True)[0]
        logger.log(prompt, gt_short, output)
    return logger.report()


# ══════════════════════════════════════════════════════════════════════════
# PrivLeak  (from muse_bench/metrics/privleak.py)
# ══════════════════════════════════════════════════════════════════════════
def compute_ppl(text, model, tokenizer, device='cuda'):
    input_ids = torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    loss, logits = outputs[:2]
    probs = torch.nn.functional.log_softmax(logits, dim=-1)
    all_prob = [probs[0, i, tok].item() for i, tok in enumerate(input_ids[0][1:])]
    return torch.exp(loss).item(), all_prob, loss.item()


def privleak_inference(text, model, tokenizer):
    pred = {}
    _, all_prob, p1 = compute_ppl(text, model, tokenizer, device=model.device)
    _, _, p_lower = compute_ppl(text.lower(), model, tokenizer, device=model.device)
    zlib_ent = len(zlib.compress(bytes(text, 'utf-8')))
    pred["PPL"] = float(p1)
    pred["PPL/lower"] = float(p1 / p_lower) if p_lower != 0 else 0.0
    pred["PPL/zlib"] = float(p1 / zlib_ent)
    for ratio in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        k = int(len(all_prob) * ratio)
        topk = np.sort(all_prob)[:k]
        pred[f"Min-{int(ratio*100)}%"] = float(-np.mean(topk).item()) if k > 0 else 0.0
    return pred


def eval_privleak(forget_data, retain_data, holdout_data, model, tokenizer):
    """Privacy leakage via membership inference AUC."""
    log = {}
    for name, data in [("forget", forget_data), ("retain", retain_data), ("holdout", holdout_data)]:
        print(f"  PrivLeak: evaluating {name} set ({len(data)} samples)...")
        log[name] = [{'text': t, **privleak_inference(t, model, tokenizer)} for t in tqdm(data, desc=name)]

    auc = {}
    ppl_types = [k for k in log['forget'][0].keys() if k != 'text']
    for s0 in ['forget', 'retain', 'holdout']:
        for s1 in ['forget', 'retain', 'holdout']:
            for pt in ppl_types:
                ppl_nm = [d[pt] for d in log[s0]]
                ppl_m  = [d[pt] for d in log[s1]]
                ppl = np.array(ppl_nm + ppl_m)
                y   = np.array([0]*len(ppl_nm) + [1]*len(ppl_m))
                fpr, tpr, _ = get_roc_curve(y, -ppl)
                auc[f"{s0}_{s1}_{pt}"] = get_auc(fpr, tpr)
    return auc, log


# ══════════════════════════════════════════════════════════════════════════
# KnowMem  (from muse_bench/metrics/knowmem.py)
# ══════════════════════════════════════════════════════════════════════════
def eval_knowmem(model, tokenizer, questions, answers,
                 icl_qs=None, icl_as=None, max_new_tokens=32):
    """Knowledge memorisation via few-shot QA."""
    if icl_qs is None: icl_qs = []
    if icl_as is None: icl_as = []
    logger = RougeEvalLogger()
    general_prompt = ""
    for q, a in zip(icl_qs, icl_as):
        general_prompt += f"Question: {q}\nAnswer: {a}\n\n"

    for question, answer in tqdm(list(zip(questions, answers)), desc="KnowMem"):
        prompt = general_prompt + f"Question: {question}\nAnswer: "
        input_ids = tokenizer(prompt, return_tensors='pt',
                              add_special_tokens=True).input_ids
        output_ids = model.generate(
            input_ids.to(model.device),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
        output_ids = output_ids[:, len(input_ids[0]):]
        output = tokenizer.batch_decode(output_ids, skip_special_tokens=True,
                                        clean_up_tokenization_spaces=True)[0]
        # Truncate at follow-up question markers
        for stop in ["\n\n", "\nQuestion", "Question:"]:
            output = output.split(stop)[0]
        logger.log(prompt, answer, output, question=question)
    return logger.report()


print("✅ MUSE evaluation metrics ready.")

## 10. Full MUSE Evaluation Pipeline
*(Adapted from `muse_bench/eval.py`)*

Passing `corpus='news'` auto-resolves all file paths from the downloaded data.

In [ ]:
# ── AUC baselines (retrained model) for PrivLeak normalisation ──────────
# Source: muse_bench/constants.py -> AUC_RETRAIN['news']
AUC_RETRAIN_NEWS = {
    "forget_holdout_Min-40%": 0.47719999999999996,
}

# ── Default data paths ──────────────────────────────────────────────────
DEFAULT_DATA_NEWS = {
    'verbmem_forget_file':       f"{DATA_DIR}/news/verbmem/forget.json",
    'privleak_forget_file':      f"{DATA_DIR}/news/privleak/forget.json",
    'privleak_retain_file':      f"{DATA_DIR}/news/privleak/retain.json",
    'privleak_holdout_file':     f"{DATA_DIR}/news/privleak/holdout.json",
    'knowmem_forget_qa_file':    f"{DATA_DIR}/news/knowmem/forget_qa.json",
    'knowmem_forget_qa_icl_file':f"{DATA_DIR}/news/knowmem/forget_qa_icl.json",
    'knowmem_retain_qa_file':    f"{DATA_DIR}/news/knowmem/retain_qa.json",
    'knowmem_retain_qa_icl_file':f"{DATA_DIR}/news/knowmem/retain_qa_icl.json",
}


def evaluate_model(
    model, tokenizer,
    metrics=('verbmem_f', 'privleak', 'knowmem_f', 'knowmem_r'),
    data=None, temp_dir=None,
    verbmem_agg_key='mean_rougeL',
    privleak_auc_key='forget_holdout_Min-40%',
    knowmem_agg_key='mean_rougeL',
    verbmem_max_new_tokens=128,
    knowmem_max_new_tokens=32,
):
    """Full MUSE evaluation on a single model.
    Source: muse_bench/eval.py – eval_model()
    """
    if data is None:
        data = DEFAULT_DATA_NEWS
    out = {}

    # 1. VerbMem (forget)
    if 'verbmem_f' in metrics:
        print("\n  Evaluating VerbMem (forget)...")
        vd = read_json(data['verbmem_forget_file'])
        agg, log = eval_verbmem(
            model, tokenizer,
            prompts=[d['prompt'] for d in vd],
            gts=[d['gt'] for d in vd],
            max_new_tokens=verbmem_max_new_tokens
        )
        if temp_dir:
            write_json(agg, os.path.join(temp_dir, "verbmem_f/agg.json"))
            write_json(log, os.path.join(temp_dir, "verbmem_f/log.json"))
        out['verbmem_f'] = agg[verbmem_agg_key] * 100
        print(f"   VerbMem = {out['verbmem_f']:.2f}%")

    # 2. PrivLeak
    if 'privleak' in metrics:
        print("\n  Evaluating PrivLeak...")
        auc_scores, log = eval_privleak(
            forget_data=read_json(data['privleak_forget_file']),
            retain_data=read_json(data['privleak_retain_file']),
            holdout_data=read_json(data['privleak_holdout_file']),
            model=model, processing_class=tokenizer
        )
        if temp_dir:
            write_json(auc_scores, os.path.join(temp_dir, "privleak/auc.json"))
            write_json(log, os.path.join(temp_dir, "privleak/log.json"))
        ref = AUC_RETRAIN_NEWS.get(privleak_auc_key, 0.5)
        out['privleak'] = (auc_scores[privleak_auc_key] - ref) / ref * 100 if ref else 0
        print(f"   PrivLeak = {out['privleak']:.2f}%")

    # 3. KnowMem (forget)
    if 'knowmem_f' in metrics:
        print("\n  Evaluating KnowMem (forget)...")
        qa = read_json(data['knowmem_forget_qa_file'])
        icl = read_json(data['knowmem_forget_qa_icl_file'])
        agg, log = eval_knowmem(
            model, tokenizer,
            questions=[d['question'] for d in qa],
            answers=[d['answer'] for d in qa],
            icl_qs=[d['question'] for d in icl],
            icl_as=[d['answer'] for d in icl],
            max_new_tokens=knowmem_max_new_tokens
        )
        if temp_dir:
            write_json(agg, os.path.join(temp_dir, "knowmem_f/agg.json"))
            write_json(log, os.path.join(temp_dir, "knowmem_f/log.json"))
        out['knowmem_f'] = agg[knowmem_agg_key] * 100
        print(f"   KnowMem (forget) = {out['knowmem_f']:.2f}%")

    # 4. KnowMem (retain)
    if 'knowmem_r' in metrics:
        print("\n  Evaluating KnowMem (retain)...")
        qa = read_json(data['knowmem_retain_qa_file'])
        icl = read_json(data['knowmem_retain_qa_icl_file'])
        agg, log = eval_knowmem(
            model, tokenizer,
            questions=[d['question'] for d in qa],
            answers=[d['answer'] for d in qa],
            icl_qs=[d['question'] for d in icl],
            icl_as=[d['answer'] for d in icl],
            max_new_tokens=knowmem_max_new_tokens
        )
        if temp_dir:
            write_json(agg, os.path.join(temp_dir, "knowmem_r/agg.json"))
            write_json(log, os.path.join(temp_dir, "knowmem_r/log.json"))
        out['knowmem_r'] = agg[knowmem_agg_key] * 100
        print(f"   KnowMem (retain) = {out['knowmem_r']:.2f}%")

    return out

print("✅ Evaluation pipeline ready.")

---
## 11. Run: Baseline Evaluation (Original Gemma 3 1B)
Evaluate the **unmodified** Gemma 3 1B to establish baselines.

In [ ]:
# Load the base model
print("Loading base Gemma 3 1B model...")
base_model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print(f"✅ Model loaded. Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

# Evaluate baseline
baseline_results = evaluate_model(base_model, tokenizer, temp_dir="./results/baseline")
print("\n" + "="*50)
print("BASELINE RESULTS (Gemma 3 1B)")
print("="*50)
for k, v in baseline_results.items():
    print(f"  {k:15s} = {v:.2f}")

In [ ]:
# Free memory before unlearning
del base_model
gc.collect()
torch.cuda.empty_cache()
print("✅ Memory cleared.")

## 12. Run: Gradient Ascent Unlearning
Perform **Gradient Ascent (GA)** unlearning — the simplest MUSE baseline.
The model *ascends* the loss on the forget set (maximises loss -> forgets).

In [ ]:
FORGET_FILE = f"{DATA_DIR}/{CORPUS}/raw/forget.txt"
RETAIN_FILE = f"{DATA_DIR}/{CORPUS}/raw/retain1.txt"

GA_OUT = "./ckpt/ga"

iterative_unlearn(
    model_dir=MODEL_NAME,
    data_file=FORGET_FILE,
    out_dir=GA_OUT,
    loss_type='ga',
    epochs=EPOCHS,
    learning_rate=LR,
    max_len=MAX_LEN,
    tokenizer_dir=MODEL_NAME,
)

In [ ]:
# Evaluate GA model
ga_model = load_model(GA_OUT)
ga_tokenizer = load_tokenizer(MODEL_NAME)

ga_results = evaluate_model(ga_model, ga_tokenizer, temp_dir="./results/ga")
print("\n" + "="*50)
print("GA UNLEARNING RESULTS")
print("="*50)
for k, v in ga_results.items():
    print(f"  {k:15s} = {v:.2f}")

del ga_model; gc.collect(); torch.cuda.empty_cache()

## 13. Run: GA + KL-Retain Unlearning
Gradient Ascent on forget set **plus** KL divergence penalty to preserve retain-set performance.

In [ ]:
GA_KLR_OUT = "./ckpt/ga_klr"

iterative_unlearn(
    model_dir=MODEL_NAME,
    data_file=FORGET_FILE,
    out_dir=GA_KLR_OUT,
    retain_data_file=RETAIN_FILE,
    loss_type='ga_klr',
    epochs=EPOCHS,
    learning_rate=LR,
    max_len=MAX_LEN,
    tokenizer_dir=MODEL_NAME,
)

In [ ]:
# Evaluate GA+KLR model
ga_klr_model = load_model(GA_KLR_OUT)
ga_klr_tokenizer = load_tokenizer(MODEL_NAME)

ga_klr_results = evaluate_model(ga_klr_model, ga_klr_tokenizer, temp_dir="./results/ga_klr")
print("\n" + "="*50)
print("GA+KLR UNLEARNING RESULTS")
print("="*50)
for k, v in ga_klr_results.items():
    print(f"  {k:15s} = {v:.2f}")

del ga_klr_model; gc.collect(); torch.cuda.empty_cache()

## 14. Run: Task Vector Unlearning
First fine-tune on the forget set, then subtract the resulting task vector from the base model.

In [ ]:
FT_OUT = "./ckpt/tv_ft"     # intermediate fine-tuned model
TV_OUT = "./ckpt/tv"        # final task-vector unlearned model

# Step 1: Fine-tune on forget data
finetune_model(
    model_dir=MODEL_NAME,
    data_file=FORGET_FILE,
    out_dir=FT_OUT,
    epochs=EPOCHS,
    learning_rate=LR,
    max_len=MAX_LEN,
    tokenizer_dir=MODEL_NAME,
)

# Step 2: Apply negated task vector
task_vector_unlearn(
    model_dir=MODEL_NAME,
    out_dir=TV_OUT,
    pt_model_dir=MODEL_NAME,
    ft_model_dir=FT_OUT,
    alpha=ALPHA_TV
)

In [ ]:
# Evaluate TV model
tv_model = load_model(TV_OUT)
tv_tokenizer = load_tokenizer(MODEL_NAME)

tv_results = evaluate_model(tv_model, tv_tokenizer, temp_dir="./results/tv")
print("\n" + "="*50)
print("TASK VECTOR UNLEARNING RESULTS")
print("="*50)
for k, v in tv_results.items():
    print(f"  {k:15s} = {v:.2f}")

del tv_model; gc.collect(); torch.cuda.empty_cache()

## 15. Post-Training Quantization (PTQ) – Recovery Analysis
Quantize each unlearned model to **4-bit** and re-evaluate to observe how much forgotten knowledge **recovers** under quantization.

In [ ]:
def load_quantized_model(model_dir, bit_width=4):
    """Load a model with bitsandbytes quantization."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=(bit_width == 4),
        load_in_8bit=(bit_width == 8),
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    return AutoModelForCausalLM.from_pretrained(
        model_dir, device_map="cuda:0",
        quantization_config=bnb_config
    )

print("✅ Quantization helper ready.")

In [ ]:
quant_results = {}
for name, ckpt_dir in [("GA", GA_OUT), ("GA+KLR", GA_KLR_OUT), ("TV", TV_OUT)]:
    print(f"\n{'='*50}")
    print(f"Quantizing {name} to 4-bit and evaluating...")
    print(f"{'='*50}")
    q_model = load_quantized_model(ckpt_dir, bit_width=4)
    q_tok   = load_tokenizer(MODEL_NAME)
    quant_results[f"{name}_4bit"] = evaluate_model(q_model, q_tok, temp_dir=f"./results/{name.lower()}_4bit")
    del q_model; gc.collect(); torch.cuda.empty_cache()

print("\n✅ Quantization evaluation complete.")

## 16. Summary – All Results

In [ ]:
# Compile results table
all_results = {
    "Baseline (FP16)": baseline_results,
    "GA (FP16)": ga_results,
    "GA+KLR (FP16)": ga_klr_results,
    "TV (FP16)": tv_results,
}
all_results.update(quant_results)

df = pd.DataFrame(all_results).T
df.index.name = "Model"
print("\n" + "="*70)
print("                 MUSE BENCHMARK RESULTS - Gemma 3 1B")
print("="*70)
print(df.to_string(float_format="%.2f"))
df.to_csv("./results/summary.csv")
print("\nSaved to ./results/summary.csv")

In [ ]:
# ── Recovery Delta (how much knowledge comes back after quantization) ──
print("\n" + "="*70)
print("           RECOVERY DELTA (4-bit - FP16)")
print("="*70)

fp16_map = {"GA": ga_results, "GA+KLR": ga_klr_results, "TV": tv_results}
deltas = {}
for name in ["GA", "GA+KLR", "TV"]:
    q_key = f"{name}_4bit"
    if q_key in quant_results:
        delta = {}
        for metric in quant_results[q_key]:
            delta[metric] = quant_results[q_key][metric] - fp16_map[name].get(metric, 0)
        deltas[f"{name} Delta(4bit-FP16)"] = delta

df_delta = pd.DataFrame(deltas).T
print(df_delta.to_string(float_format="%.2f"))
df_delta.to_csv("./results/recovery_delta.csv")
print("\nSaved to ./results/recovery_delta.csv")

---
### Done!
This notebook used the **actual MUSE benchmark** code to:
1. Load MUSE-News data from HuggingFace
2. Run 3 unlearning baselines (GA, GA+KLR, TV) on Gemma 3 1B
3. Evaluate using the official VerbMem / PrivLeak / KnowMem metrics
4. Quantize to 4-bit and measure knowledge recovery